In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from pathlib import Path
import sys
from tqdm.auto import tqdm
from scipy.spatial.distance import euclidean
from dtaidistance import dtw
import csv

project_root = Path(os.getcwd()).parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils.seed import set_seed
from src.data.preprocessing.pipeline import Pipeline
from src.data.datasets.universal_dataset import CVADataset
from src.models.network import DiffusionSSSD
from src.models.gaussian_noise import GaussianDiffusion
from src.train.trainer import setup_optimizer, DiffusionTrainer

/mnt/c/Users/edtop/ITMO/THESIS_CV_DIFF/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
CUDA extension for structured kernels (Cauchy and Vandermonde multiplication) not found. Install by going to extensions/kernels/ and running `python setup.py install`, for improved speed and memory efficiency. Note that the kernel changed for state-spaces 4.0 and must be recompiled.
Falling back on slow Cauchy and Vandermonde kernel. Install at least one of pykeops or the CUDA extension for better speed and memory efficiency.


In [2]:
EPOCHS = 40
BATCH_SIZE = 64
LR = 0.0007
WEIGHT_DECAY = 0.07
TIMESTEPS = 150
NUM_CYCLE = [1, 2, 3, 4]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Device: {DEVICE}")

fold_results_dtw = {}
fold_results_mae = {}

[*] Device: cuda


In [ ]:
NEW_RESULT_VARIABLE = '_l1_loss_+'

In [ ]:
def compute_dtw(original, reconstructed):
    """
    Dynamic Time Warping с использованием dtaidistance.
    """
    orig_f64 = np.array(original, dtype=np.float64).flatten()
    recon_f64 = np.array(reconstructed, dtype=np.float64).flatten()
    from dtaidistance import dtw
    return dtw.distance_fast(orig_f64, recon_f64, use_c=True)


test_inhibitor = "2-mercaptobenzimidazole"


RESULTS_CSV = os.path.join(project_root, "experiments", f"loio_results{NEW_RESULT_VARIABLE}.csv")

print("\n" + "="*60)
print(f"LOIO ТЕСТ | Оставляем для валидации: {test_inhibitor}")
print("="*60)

set_seed(42)

save_dir = os.path.join(project_root, "experiments", f"run_cv_{test_inhibitor}")
os.makedirs(save_dir, exist_ok=True)

pipe = Pipeline(
    num_cycle=NUM_CYCLE, 
    test_inhibitor=test_inhibitor, 
    norm_feat=True, 
    use_wavelet=False, 
    flip_the_peak=False
)

train_dataset = CVADataset(vol=pipe.train_voltage, cur=pipe.train_current, desc_df=pipe.train_analyzed_data)
val_dataset = CVADataset(vol=pipe.test_voltage, cur=pipe.test_current, desc_df=pipe.test_analyzed_data)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

num_desc_features = train_dataset[0]["features"].shape[0]
SEQ_LEN = train_dataset[0]["current"].shape[-1]

net = DiffusionSSSD(in_channels=1, desc_features=num_desc_features, base_channels=32)
diffusion = GaussianDiffusion(model=net, timesteps=TIMESTEPS).to(DEVICE)

optimizer, scheduler = setup_optimizer(model=net, lr=LR, weight_decay=WEIGHT_DECAY, epochs=EPOCHS)

trainer = DiffusionTrainer(
    diffusion_model=diffusion,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    device=DEVICE,
    save_dir=str(save_dir), 
    flip_the_peak=False
)

print(f"[*] Обучение модели без {test_inhibitor}...")
trainer.fit(epochs=EPOCHS)

print(f"[*] Запуск Инференса и расчет метрик для {test_inhibitor}...") 
diffusion.model.load_state_dict(torch.load(os.path.join(save_dir, "best_model.pth"), map_location=DEVICE)['model_state_dict'])
diffusion.eval()

dtw_scores = []
mae_scores = []

with torch.no_grad():
    for batch in tqdm(val_loader, desc=f"Eval {test_inhibitor}"):
        current = batch["current"].to(DEVICE)
        features = batch["features"].to(DEVICE)
        
        shape = (features.shape[0], 1, SEQ_LEN)
        gen_scaled = diffusion.sample(descriptors=features, shape=shape)
        
        true_np = current.squeeze(1).cpu().numpy()
        pred_np = gen_scaled.squeeze(1).cpu().numpy()
        
        for j in range(true_np.shape[0]):
            orig_signal = true_np[j]
            gen_signal = pred_np[j]
            
            mae = np.mean(np.abs(orig_signal - gen_signal))
            dtw_dist = compute_dtw(orig_signal, gen_signal)
            
            mae_scores.append(mae)
            dtw_scores.append(dtw_dist)

final_dtw = np.mean(dtw_scores)
final_mae = np.mean(mae_scores)

print(f"\nРезультат для {test_inhibitor}: DTW = {final_dtw:.4f} | MAE = {final_mae:.6f}")
file_exists = os.path.isfile(RESULTS_CSV)

with open(RESULTS_CSV, mode='a', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    if not file_exists:
        writer.writerow(['Inhibitor', 'DTW', 'MAE'])
    
    writer.writerow([test_inhibitor, final_dtw, final_mae])

print(f"[*] Данные успешно сохранены в {RESULTS_CSV}")


LOIO ТЕСТ | Оставляем для валидации: 2-mercaptobenzimidazole


Disabling PyTorch because PyTorch >= 2.4 is required but found 2.1.2+cu121
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Group 0: lr=0.0007, weight_decay=0.07, params=186881
Group 1: lr=0.0006, weight_decay=0.0, params=70400
[*] Обучение модели без 2-mercaptobenzimidazole...
Teaching on cuda...


Sampling: 100%|██████████| 150/150 [00:07<00:00, 20.12it/s]


Epoch 1 | Train Loss: 1.0229 | Val Loss: 1.4262 | LR: 0.000699 | MSE_loss 0.404429 | area_loss 0.618434 | ratio_loss 0.000000 | peaks_loss 0.000000
Saved best model (Val Loss: 1.4262)


...

Epoch 39 | Train Loss: 0.2270 | Val Loss: 0.2273 | LR: 0.000001 | MSE_loss 0.099712 | area_loss 0.127250 | ratio_loss 0.000000 | peaks_loss 0.000000


Sampling: 100%|██████████| 150/150 [00:07<00:00, 18.90it/s]


Epoch 40 | Train Loss: 0.2236 | Val Loss: 0.2257 | LR: 0.000000 | MSE_loss 0.099278 | area_loss 0.124360 | ratio_loss 0.000000 | peaks_loss 0.000000
[*] Запуск Инференса и расчет метрик для 2-mercaptobenzimidazole...


Eval 2-mercaptobenzimidazole: 100%|██████████| 13/13 [03:23<00:00, 15.68s/it]


Результат для 2-mercaptobenzimidazole: DTW = 13.0145 | MAE = 0.286977
[*] Данные успешно сохранены в /mnt/c/Users/edtop/ITMO/THESIS_CV_DIFF/experiments/loio_results_l1_loss_+.csv
